# Module 7 · Conversational interfaces: memory and context management

An LLM call is **stateless**. A chatbot feels like it remembers only because *you* send the history every time.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai openai

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) "))
# Change to "gemini-2.5-flash" if your key does not have access to the newer model.
MODEL = "gemini-3.5-flash-lite"
print('Ready. Model =', MODEL)

## 1 · No context: the model forgets

In [ ]:
for msg in ["I have 2 dogs in my home.", "How many paws are there in my home?"]:
    r = client.models.generate_content(model=MODEL, contents=msg)
    print("YOU:", msg); print("AI :", r.text.strip()[:300], "\n")

## 2 · Manual history: you resend everything

In [ ]:
history = []
def chat_manual(user_text):
    history.append(types.Content(role="user", parts=[types.Part(text=user_text)]))
    r = client.models.generate_content(model=MODEL, contents=history)
    history.append(types.Content(role="model", parts=[types.Part(text=r.text)]))
    return r.text

print(chat_manual("I have 2 dogs in my home."))
print(chat_manual("How many paws are there in my home?"))
print("messages sent on turn 2:", len(history))

## 3 · `client.chats.create` — the SDK keeps the history for you

In [ ]:
chat = client.chats.create(model=MODEL)
print(chat.send_message("I have 2 dogs in my home.").text)
print(chat.send_message("How many paws are there in my home?").text)
print("\n--- history ---")
for m in chat.get_history():
    print(m.role.upper().ljust(5), m.parts[0].text[:90].replace("\n", " "))

## 4 · Context management with summarisation
Long chats cost more and eventually overflow the context window. Strategy: when the history passes `SUMMARY_THRESHOLD` messages,
**summarise the old part** and keep only the last few turns verbatim.

Edit the numbers and re-run to see how many messages are actually sent.

In [ ]:
SUMMARY_THRESHOLD = 6      # summarise once the history is longer than this
KEEP_LAST = 2              # ... and keep this many recent messages untouched

history, summary = [], ""

def summarise(old_messages, previous_summary):
    text = "\n".join(f"{m['role']}: {m['text']}" for m in old_messages)
    prompt = (f"Previous summary: {previous_summary or '(none)'}\n\nNew conversation:\n{text}\n\n"
              "Update the summary in at most 2 sentences. Keep names, numbers and preferences.")
    return client.models.generate_content(model=MODEL, contents=prompt).text.strip()

def send(user_text):
    global history, summary
    history.append({"role": "user", "text": user_text})
    if len(history) > SUMMARY_THRESHOLD:
        old, history = history[:-KEEP_LAST], history[-KEEP_LAST:]
        summary = summarise(old, summary)
    parts = []
    if summary:
        parts.append(types.Content(role="user", parts=[types.Part(text=f"(Summary of earlier conversation: {summary})")]))
        parts.append(types.Content(role="model", parts=[types.Part(text="Understood.")]))
    for m in history:
        parts.append(types.Content(role=m["role"], parts=[types.Part(text=m["text"])]))
    r = client.models.generate_content(model=MODEL, contents=parts)
    history.append({"role": "model", "text": r.text})
    print(f"[sent {len(parts)} messages | summary: {'yes' if summary else 'no'}]")
    return r.text

for q in ["Hi, I'm Priya and I live in Pune.", "I love trekking.", "My favourite food is poha.", "I have a dog called Bruno.",
          "I work as a data analyst.", "What do you remember about me?"]:
    print("YOU:", q); print("AI :", send(q).strip()[:250], "\n")
print("Summary now:", summary)

## 5 · Multi-turn chat through the OpenAI Python library
Gemini also speaks the OpenAI protocol, so code written for `openai` works with a different `base_url` and your Gemini key.

In [ ]:
from openai import OpenAI
oa = OpenAI(api_key=os.environ["GEMINI_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
messages = [{"role": "system", "content": "You are a concise, friendly assistant."}]
for q in ["My name is Priya.", "What's my name?"]:
    messages.append({"role": "user", "content": q})
    a = oa.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content
    messages.append({"role": "assistant", "content": a})
    print("YOU:", q); print("AI :", a.strip(), "\n")